# Data Preparation for Machine Learning

This notebook provides a step-by-step guide to cleaning tree data from Grenoble in order to predict the planting year.

## Objective
Prepare the data to train a Machine Learning model that can predict missing tree planting years.

## Notebook Outline
1. Import libraries and load data  
2. Initial data exploration  
3. Data cleaning  
4. Save the data as a pickle file

## 1. Importing Libraries and Loading Data

We use several essential Python libraries for data analysis:
- pandas: for data manipulation  
- numpy: for numerical computations  
- matplotlib and seaborn: for visualization  
- sklearn: for preparing data for Machine Learning

In [ ]:
#%pip install -r '../requirements.txt'

# Libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Loading the Data
df = pd.read_csv('../arbres_grenoble_epsg4326.csv', sep=',', encoding='utf-8')

# Display the first few rows
print('Data preview:')
display(df.head())

# Dataset information
print('\nDataset information:')
display(df.info())

In [ ]:
df_traitch = df['traitementchenilles'].dropna()
df_traitch.value_counts()

In [ ]:
df['raisondeplantation'].value_counts()

In [ ]:
for i in df.columns:
    print(df[i].value_counts())

## 2. Initial Data Exploration

Before starting the cleaning process, it's crucial to understand our data:
- What columns are available?  
- What types of data do we have?  
- Are there any missing values?

In [ ]:
# Descriptive Statistics
df.dtypes
# Analysis of Missing Values
x=df.isna().sum()/len(df)*100
print(x)

### Visualisation of missing values

In [ ]:
# Creation of a heatmap to visualize missing values
#
percent_missing=pd.DataFrame()
for i in df['anneedeplantation'].value_counts().index.sort_values():
    percent_missing[i]=df[df['anneedeplantation']==int(i)].isna().sum()/len(df[df['anneedeplantation']==int(i)])*100

percent_missing

sns.heatmap(percent_missing, cmap='Greens')

In [ ]:
sns.heatmap(df.isna())

## 3. Data Cleaning

### 3.1 After the initial analysis, we will:
1. Remove rows without a planting year **'annedeplantation'**
2. Drop columns that are irrelevant for prediction  
3. Extract geographic coordinates

In [ ]:
# 1. Remove rows without a planting year **'annedeplantation'**
df = df[df['anneedeplantation'].notna()]

### 🔍 3.2 Cleaning Unnecessary Columns

We will:

- Remove columns with more than **40% missing values**  
- Remove columns that are **too specific** (e.g., unique identifiers), which don't provide useful information for machine learning  
- Remove columns that contain redundant or unhelpful information

In [ ]:
df['code_parent'].value_counts()

In [ ]:
df['code_parent_desc'].value_counts()

In [ ]:
# 2 Remove overly specific columns, such as the 'elem_point_id' column

df = df.drop(columns = ['elem_point_id', 'code', 'nom', 'categorie', 'categorie_desc','sous_categorie_desc','code_parent','code_parent_desc', 'bien_reference', 'collectivite', 'typenature', 'genre', 'genre_desc'])

In [ ]:
missing_percentage = df.isnull().mean() * 100
columns_to_drop = missing_percentage[missing_percentage > 40].index
df = df.drop(columns=columns_to_drop)

In [ ]:
# 3. Extract geographic coordinates
df[['latitude', 'longitude']] = df['geo_point_2d'].str.split(',', expand=True)
df = df.drop(columns = ['geo_point_2d'])

In [ ]:
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

In [ ]:
df.info()

In [ ]:
height_mapping = {'Moins de 10 m': 0,'de 10 m à 20 m': 1,'Plus de 20 m': 2}
df['hauteurarbre'] = df['hauteurarbre'].map(height_mapping)

In [ ]:
height_mapping = {'Arbre jeune': 0,'Arbre adulte': 1,'Arbre vieillissant': 2}
df['stadededeveloppement'] = df['stadededeveloppement'].map(height_mapping)

In [ ]:
df['adr_secteur'] = df['adr_secteur'].astype(str)

In [ ]:
df.info()

## 4. Save to data to a pickle file

In [ ]:
# Save to data to a pickle file
df.to_pickle('arbres_clean_Laura.pkl')

## Conclusion

We have completed several important cleaning steps:
1. Removed irrelevant columns or those with too many missing values  
2. Extracted geographic coordinates  

The data is now ready for the next steps of the project.